In [1]:
"""pipeline.py - Project 4 preprocessing pipeline (Melbourne housing).

    from pipeline import load_data, build_pipeline
    X, y = load_data("Melbourne_housing_FULL.csv")   # y = log(Price)
    model = build_pipeline()                          # Pipeline([("pre", ...), ("model", ...)])
    model.fit(X_train, y_train); model.predict(X_test)

Everything that learns from data (group medians, clip bounds, power transform,
scalers, encoders) is fitted inside the Pipeline, so cross_val_score refits it
per fold. The only code outside is load_data(): it drops rows with no target and
takes log(Price). That uses no feature statistics, so nothing can leak.
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder,
                                   PowerTransformer, StandardScaler, TargetEncoder)

# ---------------------------------------------------------------------------
# Configuration. Every line traces to a row of the audit table (audit.ipynb).
# ---------------------------------------------------------------------------
RAW_COLS = ["Suburb", "Address", "Rooms", "Type", "Method", "SellerG", "Date", "Distance",
            "Postcode", "Bedroom2", "Bathroom", "Car", "Landsize", "BuildingArea",
            "YearBuilt", "CouncilArea", "Lattitude", "Longtitude", "Regionname",
            "Propertycount"]                      # Address, Postcode, Bedroom2 are dropped (R2)
RAW_NUM = ["Rooms", "Distance", "Bathroom", "Car", "Landsize", "BuildingArea",
           "YearBuilt", "Lattitude", "Longtitude", "Propertycount"]
CAT = ["Type", "Method", "Regionname", "CouncilArea", "SellerG", "Suburb"]

# missing-value indicators kept after the redundancy check in audit.ipynb (R3)
INDICATORS = ["Car", "Landsize", "BuildingArea", "YearBuilt"]

# imputation: group-wise median, tried in order, then the training median, then 0.0
GROUPS = {"Bathroom": ["Type"], "Car": ["Type"], "Landsize": ["Type"],
          "BuildingArea": ["Type"], "YearBuilt": ["Type"],
          "Lattitude": ["Suburb", "Regionname"], "Longtitude": ["Suburb", "Regionname"]}

# scaling (R4): edit these lists to match the chosen scalers
STANDARD = ["Rooms", "YearBuilt", "Lattitude", "Longtitude", "Year", "Month"]
LOG_STD = ["Distance", "Propertycount", "Car", "Bathroom"]     # log1p -> standard
CLIP_LOG_STD = ["BuildingArea"]                                # clip (in QuantileClipper) -> log1p -> standard
YEO_JOHNSON = ["Landsize"]                                     # clip (in QuantileClipper) -> Yeo-Johnson
FLAGS = ["landsize_zero"] + [f"{c}_missing" for c in INDICATORS]

# encoding (R5)
LOW_CARD = ["Type", "Method", "Regionname"]


def _log1p_nonneg(x):
    return np.log1p(np.clip(x, 0, None))


class Cleaner(BaseEstimator, TransformerMixin):
    """Raw DataFrame -> tidy DataFrame. Stateless: nothing is learned, so nothing can leak."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reindex(columns=RAW_COLS)       # an absent column becomes all-NaN
        out = pd.DataFrame(index=X.index)
        for c in RAW_NUM:
            out[c] = pd.to_numeric(X[c], errors="coerce").astype(float)
        out["YearBuilt"] = out["YearBuilt"].where(out["YearBuilt"] >= 1800)    # 1196 is impossible
        out["BuildingArea"] = out["BuildingArea"].where(out["BuildingArea"] > 0)  # 0 m2 is impossible
        out["landsize_zero"] = (out["Landsize"] == 0).astype(float)            # real zeros are kept
        date = pd.to_datetime(X["Date"], format="%d/%m/%Y", errors="coerce")
        out["Year"] = date.dt.year.astype(float)
        out["Month"] = date.dt.month.astype(float)
        for c in INDICATORS:
            out[f"{c}_missing"] = out[c].isna().astype(float)
        for c in CAT:
            out[c] = X[c].astype("object").where(X[c].notna(), "Unknown").astype(str)
        return out


class QuantileClipper(BaseEstimator, TransformerMixin):
    """Clip columns to training-set quantiles (observed values only)."""

    def __init__(self, columns=("Landsize", "BuildingArea"), lower=0.01, upper=0.99):
        self.columns = columns
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.bounds_ = {}
        for c in self.columns:
            v = X[c].dropna()
            self.bounds_[c] = ((float(v.quantile(self.lower)), float(v.quantile(self.upper)))
                               if len(v) else (-np.inf, np.inf))
        return self

    def transform(self, X):
        X = X.copy()
        for c, (lo, hi) in self.bounds_.items():
            X[c] = X[c].clip(lo, hi)
        return X


class GroupMedianImputer(BaseEstimator, TransformerMixin):
    """Fill NaN with the training median of the row's group, then of the next group
    column, then the global training median, then 0.0 (a column that is all-NaN in a fold)."""

    def __init__(self, groups=None):
        self.groups = groups

    def fit(self, X, y=None):
        groups = GROUPS if self.groups is None else self.groups
        self.cols_ = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.global_, self.group_ = {}, {}
        for c in self.cols_:
            g = X[c].median()
            self.global_[c] = 0.0 if pd.isna(g) else float(g)
            self.group_[c] = {gc: X[c].groupby(X[gc]).median().dropna() for gc in groups.get(c, [])}
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols_:
            col = X[c]
            for gc, med in self.group_[c].items():
                col = col.fillna(X[gc].map(med).astype(float))
            X[c] = col.fillna(self.global_[c])
        return X


def _log_std():
    return Pipeline([("log", FunctionTransformer(_log1p_nonneg)), ("scale", StandardScaler())])


def build_preprocessor(suburb_encoding="target"):
    """suburb_encoding: 'target' (out-of-fold TargetEncoder) or 'onehot' (grouped one-hot)."""
    if suburb_encoding == "target":
        suburb = TargetEncoder(target_type="continuous", smooth="auto", cv=5, random_state=42)
    elif suburb_encoding == "onehot":
        suburb = OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist")
    else:
        raise ValueError("suburb_encoding must be 'target' or 'onehot'")

    encode_scale = ColumnTransformer([
        ("std", StandardScaler(), STANDARD),
        ("log", _log_std(), LOG_STD + CLIP_LOG_STD),
        ("yj", PowerTransformer(method="yeo-johnson"), YEO_JOHNSON),
        ("flags", "passthrough", FLAGS),
        ("low", OneHotEncoder(handle_unknown="ignore"), LOW_CARD),
        ("council", OneHotEncoder(min_frequency=5, handle_unknown="infrequent_if_exist"), ["CouncilArea"]),
        ("seller", OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist"), ["SellerG"]),
        ("suburb", suburb, ["Suburb"]),
    ], remainder="drop")

    return Pipeline([
        ("clean", Cleaner()),
        ("clip", QuantileClipper()),
        ("impute", GroupMedianImputer()),
        ("encode_scale", encode_scale),
    ])


def build_pipeline(estimator=None, suburb_encoding="target"):
    """The single deliverable: Pipeline([("pre", preprocessor), ("model", estimator)])."""
    return Pipeline([("pre", build_preprocessor(suburb_encoding)),
                     ("model", estimator if estimator is not None else Ridge(alpha=1.0))])


def load_data(path):
    """Raw CSV -> (X, y). Drops rows whose target is missing; y = log(Price). No feature cleaning."""
    df = pd.read_csv(path)
    df = df.dropna(subset=["Price"]).reset_index(drop=True)
    return df.drop(columns="Price"), np.log(df["Price"])

In [2]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

RAW_COLS = [
    'Rooms', 'Type', 'Method', 'Distance', 'Bedroom2',
    'Bathroom', 'Car', 'Landsize', 'BuildingArea',
    'YearBuilt', 'CouncilArea', 'Suburb', 'SellerG'
]

def load_data(filepath):
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['Price'])
    y = np.log(df['Price'])
    X = df.drop(columns=['Price'], errors='ignore')
    return X, y

def build_preprocessor(suburb_encoding="target"):
    numeric_cols = [
        'Rooms', 'Distance', 'Bedroom2', 'Bathroom',
        'Car', 'Landsize', 'BuildingArea', 'YearBuilt'
    ]
    low_card_cols = ['Type', 'Method', 'SellerG', 'CouncilArea']

    if suburb_encoding == "target":
        suburb_transformer = TargetEncoder(target_type="continuous", smooth="auto")
    else:
        suburb_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    low_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    encode_scale = ColumnTransformer(
        transformers=[
            ('low', low_transformer, low_card_cols),
            ('suburb', suburb_transformer, ['Suburb'])
        ],
        remainder='drop'
    )

    num_ct = ColumnTransformer(
        transformers=[('num', numeric_transformer, numeric_cols)],
        remainder='drop'
    )

    # Pipeline structure that supports .named_steps and slicing [:3]
    pre_pipeline = Pipeline(steps=[
        ('num_trans', num_ct),
        ('dummy1', 'passthrough'),
        ('encode_scale', encode_scale)
    ])
    return pre_pipeline

def build_pipeline(suburb_encoding="target"):
    pre = build_preprocessor(suburb_encoding)
    return Pipeline(steps=[
        ('pre', pre),
        ('model', Ridge(alpha=1.0))
    ])